# STAGE 9: Controlled Classical DDPG vs. Quantum QFRL Benchmark

**Objective**: Directly compare the Hybrid 4-Qubit Quantum Actor against a Classical MLP Actor under identical environment dynamics, hyperparameters, and seeds (`[42, 123, 2026]`).

**Hardware Required**: Kaggle GPU (Tesla T4) enabled.

In [ ]:
# 1. Setup Working Directory and Pull/Clone Repository
import os, sys, shutil

os.chdir('/kaggle/working')
repo_dir = '/kaggle/working/QRFL'
if not os.path.exists(repo_dir):
    print('Cloning QRFL repository...')
    !git clone https://github.com/toqeer-ahmed/QRFL.git
else:
    print('Repository exists, updating...')
    %cd /kaggle/working/QRFL
    !git pull
    %cd /kaggle/working

qfrl_dir = os.path.join(repo_dir, 'QFRL_Highway')
os.chdir(qfrl_dir)
if qfrl_dir not in sys.path:
    sys.path.insert(0, qfrl_dir)

# 2. Install Required Quantum & Support Libraries
!pip install -q --no-deps torchpack torchdiffeq
!pip install -q qiskit-ibm-runtime qiskit-aer loguru multimethod tensorpack toml
!pip install -q --no-deps git+https://github.com/mit-han-lab/torchquantum.git
print('Dependencies installed successfully.')


In [ ]:
# 3. Ensure Latest Stage 9 Files Are Present
classical_actor_content = 'import os\nimport torch\nimport torch.nn as nn\nimport torch.optim as optim\nimport numpy as np\n\n# Note: We import ONLY the classical Critic from rl.ddpg.\n# We do NOT import QuantumActor, torchquantum, or any quantum components.\nfrom rl.ddpg import Critic\n\nclass ClassicalActor(nn.Module):\n    """\n    Classical MLP Actor for QFRL Benchmark.\n    \n    Architecture:\n    Maps 150-dimensional environment state -> Hidden Layers -> 5-dimensional continuous action in [-1, 1].\n    Uses Tanh activation on the final layer to guarantee outputs are strictly within [-1.0, 1.0].\n    \n    Default Architecture:\n    Linear(150, 64) -> ReLU -> Linear(64, 64) -> ReLU -> Linear(64, 5) -> Tanh\n    """\n    def __init__(self, state_dim=150, action_dim=5, hidden_dims=(64, 64)):\n        super(ClassicalActor, self).__init__()\n        self.state_dim = state_dim\n        self.action_dim = action_dim\n        self.hidden_dims = list(hidden_dims)\n        \n        layers = []\n        prev_dim = state_dim\n        for h_dim in self.hidden_dims:\n            layers.append(nn.Linear(prev_dim, h_dim))\n            layers.append(nn.ReLU())\n            prev_dim = h_dim\n            \n        layers.append(nn.Linear(prev_dim, action_dim))\n        layers.append(nn.Tanh())\n        \n        self.network = nn.Sequential(*layers)\n        \n    def forward(self, state: torch.Tensor) -> torch.Tensor:\n        """\n        Forward pass.\n        Args:\n            state (torch.Tensor): Tensor of shape (batch_size, state_dim) or (state_dim,)\n        Returns:\n            torch.Tensor: Action tensor of shape (batch_size, action_dim) bounded in [-1.0, 1.0]\n        """\n        return self.network(state)\n\nclass ClassicalDDPGAgent:\n    """\n    Classical DDPG Agent using ClassicalActor and Classical Critic.\n    Maintains 100% interface compatibility with DDPGAgent while ensuring\n    zero quantum dependencies or quantum execution overhead.\n    """\n    def __init__(self, state_dim=150, action_dim=5, config=None):\n        self.config = config or {}\n        self.device = torch.device(self.config.get(\'device\', \'cpu\'))\n        self.state_dim = state_dim\n        self.action_dim = action_dim\n        \n        # Hyperparameters (matched to paper and quantum baseline)\n        self.gamma = self.config.get(\'gamma\', 0.95)\n        self.actor_lr = self.config.get(\'actor_lr\', 0.001)\n        self.critic_lr = self.config.get(\'critic_lr\', 0.0025)\n        self.tau = self.config.get(\'tau\', 0.005)\n        self.exploration_noise = self.config.get(\'exploration_noise\', 0.1)\n        \n        hidden_dims = self.config.get(\'classical_actor_hidden\', (64, 64))\n        \n        # Initialize Classical Actor & Targets\n        self.actor = ClassicalActor(state_dim=state_dim, action_dim=action_dim, hidden_dims=hidden_dims).to(self.device)\n        self.actor_target = ClassicalActor(state_dim=state_dim, action_dim=action_dim, hidden_dims=hidden_dims).to(self.device)\n        self.actor_target.load_state_dict(self.actor.state_dict())\n        \n        # Initialize Classical Critic & Targets (identical to quantum baseline)\n        critic_h1 = self.config.get(\'critic_hidden1\', 64)\n        critic_h2 = self.config.get(\'critic_hidden2\', 64)\n        self.critic = Critic(state_dim=state_dim, action_dim=action_dim, hidden_dim1=critic_h1, hidden_dim2=critic_h2).to(self.device)\n        self.critic_target = Critic(state_dim=state_dim, action_dim=action_dim, hidden_dim1=critic_h1, hidden_dim2=critic_h2).to(self.device)\n        self.critic_target.load_state_dict(self.critic.state_dict())\n        \n        # Optimizers (identical settings to quantum baseline)\n        self.actor_optimizer = optim.Adam(self.actor.parameters(), lr=self.actor_lr)\n        self.critic_optimizer = optim.Adam(self.critic.parameters(), lr=self.critic_lr)\n        self.critic_criterion = nn.MSELoss()\n        \n    def select_action(self, state: np.ndarray, add_noise=True, noise_scale=None) -> np.ndarray:\n        """\n        Selects a continuous action in [-1.0, 1.0].\n        Args:\n            state (np.ndarray): Environment state of shape (150,)\n            add_noise (bool): Whether to add Gaussian exploration noise\n            noise_scale (float, optional): Custom noise standard deviation\n        Returns:\n            np.ndarray: Action vector of shape (5,) in [-1.0, 1.0]\n        """\n        state_tensor = torch.FloatTensor(state).unsqueeze(0).to(self.device)\n        with torch.no_grad():\n            action = self.actor(state_tensor).cpu().numpy()[0]\n            \n        if add_noise:\n            scale = noise_scale if noise_scale is not None else self.exploration_noise\n            noise = np.random.normal(0, scale, size=action.shape)\n            action = action + noise\n            \n        return np.clip(action, -1.0, 1.0)\n        \n    def train(self, replay_buffer, batch_size=32):\n        """\n        Performs one gradient update step of DDPG.\n        Returns:\n            tuple: (actor_loss, critic_loss) or (None, None) if insufficient samples\n        """\n        if len(replay_buffer) < batch_size:\n            return None, None\n            \n        batch = replay_buffer.sample(batch_size)\n        \n        state = torch.FloatTensor(batch[\'states\']).to(self.device)\n        action = torch.FloatTensor(batch[\'actions\']).to(self.device)\n        reward = torch.FloatTensor(batch[\'rewards\']).to(self.device)\n        next_state = torch.FloatTensor(batch[\'next_states\']).to(self.device)\n        done = torch.FloatTensor(batch[\'dones\']).to(self.device)\n        \n        # Critic Update\n        with torch.no_grad():\n            next_action = self.actor_target(next_state)\n            target_q = self.critic_target(next_state, next_action)\n            target_q = reward + (1 - done) * self.gamma * target_q\n            \n        current_q = self.critic(state, action)\n        critic_loss = self.critic_criterion(current_q, target_q)\n        \n        self.critic_optimizer.zero_grad()\n        critic_loss.backward()\n        self.critic_optimizer.step()\n        \n        # Actor Update\n        actor_loss = -self.critic(state, self.actor(state)).mean()\n        \n        self.actor_optimizer.zero_grad()\n        actor_loss.backward()\n        self.actor_optimizer.step()\n        \n        # Soft Target Updates\n        for param, target_param in zip(self.actor.parameters(), self.actor_target.parameters()):\n            target_param.data.copy_(self.tau * param.data + (1 - self.tau) * target_param.data)\n            \n        for param, target_param in zip(self.critic.parameters(), self.critic_target.parameters()):\n            target_param.data.copy_(self.tau * param.data + (1 - self.tau) * target_param.data)\n            \n        return actor_loss.item(), critic_loss.item()\n        \n    def save_checkpoint(self, path: str, episode: int, seed: int):\n        """Saves classical agent checkpoint."""\n        torch.save({\n            \'episode\': episode,\n            \'seed\': seed,\n            \'actor_type\': \'classical\',\n            \'actor_state_dict\': self.actor.state_dict(),\n            \'critic_state_dict\': self.critic.state_dict(),\n            \'actor_target_state_dict\': self.actor_target.state_dict(),\n            \'critic_target_state_dict\': self.critic_target.state_dict(),\n            \'actor_optimizer_state_dict\': self.actor_optimizer.state_dict(),\n            \'critic_optimizer_state_dict\': self.critic_optimizer.state_dict(),\n            \'config\': self.config\n        }, path)\n        \n    def load_checkpoint(self, path: str):\n        """Loads classical agent checkpoint."""\n        checkpoint = torch.load(path, map_location=self.device)\n        self.actor.load_state_dict(checkpoint[\'actor_state_dict\'])\n        self.critic.load_state_dict(checkpoint[\'critic_state_dict\'])\n        self.actor_target.load_state_dict(checkpoint[\'actor_target_state_dict\'])\n        self.critic_target.load_state_dict(checkpoint[\'critic_target_state_dict\'])\n        self.actor_optimizer.load_state_dict(checkpoint[\'actor_optimizer_state_dict\'])\n        self.critic_optimizer.load_state_dict(checkpoint[\'critic_optimizer_state_dict\'])\n        return checkpoint[\'episode\'], checkpoint[\'seed\'], checkpoint[\'config\']\n'
compare_content = 'import os\nimport sys\nimport time\nimport copy\nimport types\nimport random\nimport argparse\nimport yaml\nimport csv\nimport numpy as np\nimport torch\nimport torch.nn as nn\nimport matplotlib\nmatplotlib.use(\'Agg\')\nimport matplotlib.pyplot as plt\n\n# Ensure package imports work regardless of execution location\nsys.path.insert(0, os.path.abspath(os.path.join(os.path.dirname(__file__), \'..\')))\n\n# Mock qiskit_ibm_runtime if absent to prevent import failure in torchquantum\ntry:\n    import qiskit_ibm_runtime\nexcept ImportError:\n    sys.modules[\'qiskit_ibm_runtime\'] = types.ModuleType(\'qiskit_ibm_runtime\')\n\nfrom environment.environment import QFRLEnvironment\nfrom rl.ddpg import DDPGAgent\nfrom rl.classical_actor import ClassicalDDPGAgent, ClassicalActor\nfrom rl.replay_buffer import ReplayBuffer\n\ndef set_seed(seed: int):\n    """Set random seed across all libraries for deterministic execution."""\n    random.seed(seed)\n    np.random.seed(seed)\n    torch.manual_seed(seed)\n    if torch.cuda.is_available():\n        torch.cuda.manual_seed(seed)\n        torch.cuda.manual_seed_all(seed)\n        torch.backends.cudnn.deterministic = True\n        torch.backends.cudnn.benchmark = False\n\ndef count_trainable_parameters(model: nn.Module) -> int:\n    """Counts requires_grad parameters in a PyTorch module."""\n    return sum(p.numel() for p in model.parameters() if p.requires_grad)\n\ndef verify_quantum_backend(allow_mock: bool = False):\n    """\n    Verifies that the real TorchQuantum backend is loaded from site-packages.\n    Fails loudly if mock_torchquantum is imported without explicit permission.\n    """\n    print("\\n" + "=" * 60)\n    print("BACKEND VERIFICATION (QUANTUM MODEL)")\n    print("=" * 60)\n    print(f"Python version      : {sys.version.split()[0]} ({sys.platform})")\n    print(f"PyTorch version     : {torch.__version__}")\n    cuda_avail = torch.cuda.is_available()\n    print(f"CUDA Available      : {cuda_avail}")\n    if cuda_avail:\n        print(f"GPU Device Name     : {torch.cuda.get_device_name(0)}")\n        print(f"GPU Device Count    : {torch.cuda.device_count()}")\n    else:\n        print("GPU Device Name     : N/A (Running on CPU)")\n\n    # Ensure torchpack and torchdiffeq are available for torchquantum\n    try:\n        import torchpack\n    except ImportError:\n        try:\n            import subprocess\n            print("[INFO] Installing missing TorchQuantum utility: torchpack & torchdiffeq...")\n            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "torchpack", "torchdiffeq"])\n            print("[PASS] torchpack installed successfully.")\n        except Exception as e:\n            print(f"[WARN] Auto-install of torchpack failed: {e}")\n\n    try:\n        import torchquantum as tq\n        backend_path = str(getattr(tq, \'__file__\', \'\'))\n        version = str(getattr(tq, \'__version__\', \'0.3.0\'))\n        \n        is_mock = "mock_torchquantum" in backend_path or ("QFRL_Highway" in backend_path and "mock" in backend_path)\n        \n        if is_mock:\n            backend_name = f"TorchQuantum {version} (Mock)"\n            print(f"TorchQuantum version: {version}")\n            print(f"TorchQuantum path   : {backend_path}")\n            if not allow_mock:\n                err_msg = (\n                    "FATAL ERROR: Loaded local mock_torchquantum instead of real TorchQuantum!\\n"\n                    f"Path: {backend_path}\\n"\n                    "Stage 9 benchmark requires real TorchQuantum backend or explicit --allow_mock flag."\n                )\n                print(f"[FAIL] {err_msg}")\n                raise RuntimeError(err_msg)\n            else:\n                print(f"[WARN] Local mock permitted via --allow_mock flag.")\n        else:\n            backend_name = f"TorchQuantum {version} (Real)"\n            print(f"TorchQuantum version: {version}")\n            print(f"TorchQuantum path   : {backend_path}")\n            print(f"[PASS] Real TorchQuantum backend confirmed.")\n            \n        print("=" * 60 + "\\n")\n        return tq, backend_name, is_mock\n    except ImportError as e:\n        if allow_mock:\n            try:\n                import mock_torchquantum as tq\n                backend_name = "TorchQuantum (Mock)"\n                print(f"[WARN] Imported mock_torchquantum under --allow_mock flag.")\n                return tq, backend_name, True\n            except ImportError:\n                pass\n        err_msg = f"FATAL ERROR: Failed to import torchquantum: {e}"\n        print(f"[FAIL] {err_msg}")\n        raise RuntimeError(err_msg)\n\ndef run_pilot_smoke_test(env_config, models=None, device="cpu", allow_mock=False):\n    """\n    Executes short 2-episode pilot per model to verify that:\n    1. Classical model uses only ClassicalActor (no quantum imports or calls).\n    2. Quantum model uses TorchQuantum properly.\n    3. Both perform valid updates, finite losses, valid actions, and bit-exact checkpointing.\n    """\n    if models is None:\n        models = ["classical", "quantum"]\n        \n    print("=" * 70)\n    print(f"STAGE 9 PILOT SMOKE TEST: VERIFYING AGENTS {models}")\n    print("=" * 70)\n    \n    set_seed(42)\n    env = QFRLEnvironment()\n    env.initialize_from_config(env_config)\n    \n    c_config = copy.deepcopy(env_config)\n    c_config[\'device\'] = device\n    \n    # 1. Classical Model Verification\n    if "classical" in models:\n        print("1. Testing Classical Agent...")\n        c_agent = ClassicalDDPGAgent(state_dim=150, action_dim=5, config=c_config)\n        \n        c_actor_params = count_trainable_parameters(c_agent.actor)\n        c_critic_params = count_trainable_parameters(c_agent.critic)\n        print(f"   [PASS] ClassicalActor instantiated. Actor params: {c_actor_params:,}, Critic params: {c_critic_params:,}")\n    \n    c_buffer = ReplayBuffer(state_dim=150, action_dim=5, max_size=100)\n    s = env.reset()\n    for _ in range(35):\n        a = c_agent.select_action(s)\n        s_next, r, d, info = env.step(a)\n        c_buffer.add(s, a, r, s_next, d)\n        s = s_next\n    \n    c_al, c_cl = c_agent.train(c_buffer, batch_size=32)\n    assert c_al is not None and c_cl is not None, "Classical train returned None"\n    assert np.isfinite(c_al) and np.isfinite(c_cl), "Classical losses not finite"\n    print(f"   [PASS] Classical DDPG update verified. ALoss: {c_al:.4f}, CLoss: {c_cl:.4e}")\n    \n    # Test Classical Checkpoint\n    os.makedirs("results/checkpoints", exist_ok=True)\n    c_ckpt_path = "results/checkpoints/pilot_classical_test.pt"\n    c_agent.save_checkpoint(c_ckpt_path, episode=1, seed=42)\n    c_agent_restored = ClassicalDDPGAgent(state_dim=150, action_dim=5, config=c_config)\n    c_agent_restored.load_checkpoint(c_ckpt_path)\n    for p1, p2 in zip(c_agent.actor.parameters(), c_agent_restored.actor.parameters()):\n        assert torch.allclose(p1, p2), "Classical checkpoint restoration failed"\n    print("   [PASS] Classical checkpoint save and bit-exact restore verified.")\n    if os.path.exists(c_ckpt_path): os.remove(c_ckpt_path)\n\n    # 2. Quantum Model Verification\n    if "quantum" in models:\n        print("\\n2. Testing Quantum Agent...")\n        verify_quantum_backend(allow_mock=allow_mock)\n        q_agent = DDPGAgent(config=c_config)\n        q_actor_params = count_trainable_parameters(q_agent.actor)\n        q_critic_params = count_trainable_parameters(q_agent.critic)\n        print(f"   [PASS] QuantumActor instantiated. Actor params: {q_actor_params:,}, Critic params: {q_critic_params:,}")\n        \n        q_buffer = ReplayBuffer(state_dim=150, action_dim=5, max_size=100)\n        s = env.reset()\n        for _ in range(35):\n            a = q_agent.select_action(s)\n            s_next, r, d, info = env.step(a)\n            q_buffer.add(s, a, r, s_next, d)\n            s = s_next\n            \n        q_al, q_cl = q_agent.train(q_buffer, batch_size=32)\n        assert q_al is not None and q_cl is not None, "Quantum train returned None"\n        assert np.isfinite(q_al) and np.isfinite(q_cl), "Quantum losses not finite"\n        print(f"   [PASS] Quantum DDPG update verified. ALoss: {q_al:.4f}, CLoss: {q_cl:.4e}")\n        \n        # Test Quantum Checkpoint\n        q_ckpt_path = "results/checkpoints/pilot_quantum_test.pt"\n        q_agent.save_checkpoint(q_ckpt_path, episode=1, seed=42)\n        q_agent_restored = DDPGAgent(config=c_config)\n        q_agent_restored.load_checkpoint(q_ckpt_path)\n        for p1, p2 in zip(q_agent.actor.parameters(), q_agent_restored.actor.parameters()):\n            assert torch.allclose(p1, p2), "Quantum checkpoint restoration failed"\n        print("   [PASS] Quantum checkpoint save and bit-exact restore verified.")\n        if os.path.exists(q_ckpt_path): os.remove(q_ckpt_path)\n\n    print("=" * 70)\n    print("STAGE 9 PILOT SMOKE TEST RESULT: PASS")\n    print("=" * 70 + "\\n")\n    return True\n\ndef run_controlled_comparison(\n    config_path: str,\n    models: list = ["classical", "quantum"],\n    episodes: int = 15,\n    steps_per_episode: int = 50,\n    seeds: list = [42, 123, 2026],\n    device: str = "cpu",\n    output_dir: str = "results",\n    allow_mock: bool = False,\n    noise_schedule: str = "constant",\n    decay_rate: float = 0.95\n):\n    """\n    Executes matched, controlled comparison between Classical and Quantum DDPG models.\n    """\n    with open(config_path, \'r\') as f:\n        config = yaml.safe_load(f)\n        \n    os.makedirs(output_dir, exist_ok=True)\n    ckpt_dir = os.path.join(output_dir, "checkpoints")\n    os.makedirs(ckpt_dir, exist_ok=True)\n    \n    # Verify backend if quantum is included\n    backend_name = "Classical PyTorch"\n    if "quantum" in models:\n        _, q_backend, _ = verify_quantum_backend(allow_mock=allow_mock)\n    else:\n        q_backend = "N/A"\n        \n    csv_path = os.path.join(output_dir, "quantum_classical_comparison.csv")\n    csv_headers = [\n        "model_type",\n        "seed",\n        "episode",\n        "episode_reward",\n        "mean_step_reward",\n        "energy_efficiency_mbits_per_joule",\n        "completed_bits",\n        "total_energy",\n        "mean_latency",\n        "task_completion_ratio",\n        "actor_loss",\n        "critic_loss",\n        "actor_trainable_params",\n        "critic_trainable_params",\n        "inference_time_ms",\n        "duration_seconds",\n        "noise_schedule",\n        "current_noise_scale",\n        "backend_name"\n    ]\n    \n    file_exists = os.path.exists(csv_path)\n    csv_file = open(csv_path, \'a\' if file_exists else \'w\', newline=\'\')\n    writer = csv.DictWriter(csv_file, fieldnames=csv_headers)\n    if not file_exists:\n        writer.writeheader()\n        \n    all_results = {m: {s: [] for s in seeds} for m in models}\n    \n    buffer_size = config.get(\'rl\', {}).get(\'replay_buffer_size\', 6000)\n    batch_size = config.get(\'rl\', {}).get(\'batch_size\', 32)\n    warmup_steps = 32\n    initial_noise = config.get(\'rl\', {}).get(\'exploration_noise\', 0.1)\n    \n    print("\\n" + "=" * 70)\n    print("STARTING CONTROLLED QFRL BENCHMARK")\n    print(f"Models to evaluate   : {models}")\n    print(f"Seeds               : {seeds}")\n    print(f"Episodes per seed   : {episodes}")\n    print(f"Steps per episode   : {steps_per_episode}")\n    print(f"Exploration schedule: {noise_schedule}")\n    print(f"Device              : {device}")\n    print(f"Output CSV          : {csv_path}")\n    print("=" * 70 + "\\n")\n    \n    model_stats = {}\n    \n    for model_name in models:\n        print("\\n" + "#" * 70)\n        print(f"MODEL EVALUATION: {model_name.upper()}")\n        print("#" * 70)\n        \n        m_start_time = time.time()\n        \n        for seed in seeds:\n            print("=" * 60)\n            print(f"[{model_name.upper()}] Starting Run — Seed {seed}")\n            print("=" * 60)\n            set_seed(seed)\n            \n            env = QFRLEnvironment()\n            env.initialize_from_config(config)\n            \n            run_cfg = copy.deepcopy(config)\n            run_cfg[\'device\'] = device\n            \n            if model_name == "classical":\n                agent = ClassicalDDPGAgent(state_dim=150, action_dim=5, config=run_cfg)\n                curr_backend = "Classical PyTorch MLP"\n            elif model_name == "quantum":\n                agent = DDPGAgent(config=run_cfg)\n                curr_backend = q_backend\n            else:\n                raise ValueError(f"Unknown model name: {model_name}")\n                \n            actor_params = count_trainable_parameters(agent.actor)\n            critic_params = count_trainable_parameters(agent.critic)\n            \n            replay_buffer = ReplayBuffer(state_dim=150, action_dim=5, max_size=buffer_size)\n            \n            # Replay buffer warm-up\n            w_state = env.reset()\n            for _ in range(warmup_steps):\n                w_action = agent.select_action(w_state, add_noise=True)\n                w_next, w_r, w_d, w_info = env.step(w_action)\n                replay_buffer.add(w_state, w_action, w_r, w_next, w_d)\n                w_state = w_next\n                if w_d: w_state = env.reset()\n                \n            current_noise = initial_noise\n            \n            for ep in range(1, episodes + 1):\n                ep_start = time.time()\n                state = env.reset()\n                \n                ep_reward = 0.0\n                ep_completed_bits = 0.0\n                ep_total_energy = 0.0\n                ep_latencies = []\n                ep_comp_ratios = []\n                ep_actor_losses = []\n                ep_critic_losses = []\n                inference_times = []\n                \n                # Noise decay logic\n                if noise_schedule == "decay":\n                    current_noise = max(0.01, initial_noise * (decay_rate ** (ep - 1)))\n                else:\n                    current_noise = initial_noise\n                    \n                for step in range(steps_per_episode):\n                    t0 = time.time()\n                    try:\n                        action = agent.select_action(state, add_noise=True, noise_scale=current_noise)\n                    except TypeError:\n                        if hasattr(agent, \'exploration_noise\'):\n                            agent.exploration_noise = current_noise\n                        action = agent.select_action(state, add_noise=True)\n                    t_inf = (time.time() - t0) * 1000.0\n                    inference_times.append(t_inf)\n                    \n                    next_state, reward, done, info = env.step(action)\n                    replay_buffer.add(state, action, reward, next_state, done)\n                    \n                    al, cl = agent.train(replay_buffer, batch_size=batch_size)\n                    if al is not None and cl is not None:\n                        ep_actor_losses.append(al)\n                        ep_critic_losses.append(cl)\n                        \n                    state = next_state\n                    ep_reward += reward\n                    ep_completed_bits += info.get(\'completed_bits\', 0.0)\n                    ep_total_energy += info.get(\'total_energy\', 0.0)\n                    ep_comp_ratios.append(info.get(\'task_completion_ratio\', 1.0))\n                    \n                    tx_time = info.get(\'transmission_time\', 0.0)\n                    cp_time = info.get(\'computation_time\', 0.0)\n                    ep_latencies.append(max(tx_time, cp_time))\n                    \n                    if done: break\n                    \n                ep_duration = time.time() - ep_start\n                mean_step_reward = ep_reward / max(step + 1, 1)\n                ee_mbits_per_joule = (ep_completed_bits / max(ep_total_energy, 1e-6)) / 1e6\n                avg_al = float(np.mean(ep_actor_losses)) if ep_actor_losses else float(\'nan\')\n                avg_cl = float(np.mean(ep_critic_losses)) if ep_critic_losses else float(\'nan\')\n                mean_latency = float(np.mean(ep_latencies)) if ep_latencies else 0.0\n                mean_comp_ratio = float(np.mean(ep_comp_ratios)) if ep_comp_ratios else 0.0\n                mean_inf = float(np.mean(inference_times)) if inference_times else 0.0\n                \n                row = {\n                    "model_type": model_name,\n                    "seed": seed,\n                    "episode": ep,\n                    "episode_reward": round(ep_reward, 4),\n                    "mean_step_reward": round(mean_step_reward, 4),\n                    "energy_efficiency_mbits_per_joule": round(ee_mbits_per_joule, 6),\n                    "completed_bits": round(ep_completed_bits, 2),\n                    "total_energy": round(ep_total_energy, 4),\n                    "mean_latency": round(mean_latency, 6),\n                    "task_completion_ratio": round(mean_comp_ratio, 4),\n                    "actor_loss": round(avg_al, 4) if np.isfinite(avg_al) else "nan",\n                    "critic_loss": round(avg_cl, 4) if np.isfinite(avg_cl) else "nan",\n                    "actor_trainable_params": actor_params,\n                    "critic_trainable_params": critic_params,\n                    "inference_time_ms": round(mean_inf, 3),\n                    "duration_seconds": round(ep_duration, 2),\n                    "noise_schedule": noise_schedule,\n                    "current_noise_scale": round(current_noise, 4),\n                    "backend_name": curr_backend\n                }\n                \n                writer.writerow(row)\n                csv_file.flush()\n                all_results[model_name][seed].append(row)\n                \n                print(\n                    f"[{model_name[:5].upper()}] Ep {ep:2d}/{episodes:2d} | "\n                    f"Reward: {ep_reward:9.2f} | "\n                    f"EE: {ee_mbits_per_joule:6.3f} Mb/J | "\n                    f"Energy: {ep_total_energy:7.2f} J | "\n                    f"ALoss: {avg_al:8.2f} | "\n                    f"CLoss: {avg_cl:10.2e} | "\n                    f"Inf: {mean_inf:4.1f}ms | "\n                    f"Time: {ep_duration:4.1f}s"\n                )\n                \n                if ep % 5 == 0 or ep == episodes:\n                    ckpt_file = os.path.join(ckpt_dir, f"{model_name}_seed{seed}_ep{ep}.pt")\n                    agent.save_checkpoint(ckpt_file, episode=ep, seed=seed)\n                    \n        model_stats[model_name] = {\n            \'total_time\': time.time() - m_start_time,\n            \'actor_params\': actor_params,\n            \'critic_params\': critic_params\n        }\n\n    csv_file.close()\n    print("\\nBenchmark CSV saved to:", csv_path)\n    \n    # Generate Comparison Plot\n    plot_path = os.path.join(output_dir, "quantum_classical_comparison.png")\n    generate_comparison_plot(all_results, models, seeds, episodes, plot_path)\n    \n    # Print Statistical Summary\n    print_statistical_summary(all_results, models, seeds, model_stats)\n    return all_results\n\ndef generate_comparison_plot(all_results, models, seeds, episodes, plot_path):\n    """Generates 4-panel publication-grade comparison plot."""\n    fig, axes = plt.subplots(2, 2, figsize=(14, 10))\n    fig.suptitle("Controlled Comparison: Classical vs. Quantum QFRL (Matched Seeds)", fontsize=16, fontweight=\'bold\')\n    \n    colors = {\'classical\': \'#1f77b4\', \'quantum\': \'#ff7f0e\'}\n    \n    # 1. Energy Efficiency\n    ax = axes[0, 0]\n    for m in models:\n        # Collect per-episode across seeds\n        matrix = np.array([[row[\'energy_efficiency_mbits_per_joule\'] for row in all_results[m][s]] for s in seeds])\n        mean_ee = np.mean(matrix, axis=0)\n        std_ee = np.std(matrix, axis=0)\n        eps = np.arange(1, episodes + 1)\n        ax.plot(eps, mean_ee, label=f"{m.capitalize()} (Mean)", color=colors.get(m, \'blue\'), lw=2.5)\n        ax.fill_between(eps, mean_ee - std_ee, mean_ee + std_ee, color=colors.get(m, \'blue\'), alpha=0.15)\n    ax.set_title("Energy Efficiency (Mb/J) Across Episodes", fontweight=\'bold\')\n    ax.set_xlabel("Episode")\n    ax.set_ylabel("Energy Efficiency (Mb/J)")\n    ax.grid(True, alpha=0.3)\n    ax.legend()\n    \n    # 2. Total Energy Consumption\n    ax = axes[0, 1]\n    for m in models:\n        matrix = np.array([[row[\'total_energy\'] for row in all_results[m][s]] for s in seeds])\n        mean_en = np.mean(matrix, axis=0)\n        std_en = np.std(matrix, axis=0)\n        eps = np.arange(1, episodes + 1)\n        ax.plot(eps, mean_en, label=f"{m.capitalize()}", color=colors.get(m, \'blue\'), lw=2.5)\n        ax.fill_between(eps, mean_en - std_en, mean_en + std_en, color=colors.get(m, \'blue\'), alpha=0.15)\n    ax.set_title("Total Energy Consumption (Joules)", fontweight=\'bold\')\n    ax.set_xlabel("Episode")\n    ax.set_ylabel("Energy (Joules)")\n    ax.grid(True, alpha=0.3)\n    ax.legend()\n    \n    # 3. Episode Reward\n    ax = axes[1, 0]\n    for m in models:\n        matrix = np.array([[row[\'episode_reward\'] for row in all_results[m][s]] for s in seeds])\n        mean_r = np.mean(matrix, axis=0)\n        std_r = np.std(matrix, axis=0)\n        eps = np.arange(1, episodes + 1)\n        ax.plot(eps, mean_r, label=f"{m.capitalize()}", color=colors.get(m, \'blue\'), lw=2.5)\n        ax.fill_between(eps, mean_r - std_r, mean_r + std_r, color=colors.get(m, \'blue\'), alpha=0.15)\n    ax.set_title("Episode Reward Across Episodes", fontweight=\'bold\')\n    ax.set_xlabel("Episode")\n    ax.set_ylabel("Reward")\n    ax.grid(True, alpha=0.3)\n    ax.legend()\n    \n    # 4. Critic Loss\n    ax = axes[1, 1]\n    for m in models:\n        matrix = np.array([[row[\'critic_loss\'] for row in all_results[m][s]] for s in seeds])\n        mean_cl = np.mean(matrix, axis=0)\n        eps = np.arange(1, episodes + 1)\n        ax.plot(eps, mean_cl, label=f"{m.capitalize()} Critic Loss", color=colors.get(m, \'blue\'), lw=2)\n    ax.set_title("Critic Loss (MSE)", fontweight=\'bold\')\n    ax.set_xlabel("Episode")\n    ax.set_ylabel("Critic Loss")\n    ax.set_yscale(\'log\')\n    ax.grid(True, alpha=0.3)\n    ax.legend()\n    \n    plt.tight_layout()\n    plt.savefig(plot_path, dpi=300)\n    print(f"Generated comparison plot: {plot_path}")\n\ndef print_statistical_summary(all_results, models, seeds, model_stats):\n    """Prints comprehensive Markdown-compatible statistical summary table."""\n    print("\\n" + "=" * 80)\n    print("STAGE 9 BENCHMARK STATISTICAL SUMMARY")\n    print("=" * 80)\n    \n    header = "| Metric / Model | " + " | ".join([f"{m.capitalize()}" for m in models]) + " |"\n    sep = "|---| " + " | ".join(["---" for _ in models]) + " |"\n    print(header)\n    print(sep)\n    \n    # Trainable parameters\n    p_row = "| **Actor Parameters** | " + " | ".join([f"{model_stats[m][\'actor_params\']:,}" for m in models]) + " |"\n    print(p_row)\n    c_row = "| **Critic Parameters** | " + " | ".join([f"{model_stats[m][\'critic_params\']:,}" for m in models]) + " |"\n    print(c_row)\n    t_row = "| **Total Runtime** | " + " | ".join([f"{model_stats[m][\'total_time\']:.1f}s" for m in models]) + " |"\n    print(t_row)\n    \n    # Summary per model\n    means_ee = {}\n    for m in models:\n        ee_vals = [row[\'energy_efficiency_mbits_per_joule\'] for s in seeds for row in all_results[m][s]]\n        means_ee[m] = (np.mean(ee_vals), np.std(ee_vals))\n        \n    ee_row = "| **Mean Energy Efficiency (Mb/J)** | " + " | ".join([f"{means_ee[m][0]:.4f} +/- {means_ee[m][1]:.4f}" for m in models]) + " |"\n    print(ee_row)\n    \n    means_en = {}\n    for m in models:\n        en_vals = [row[\'total_energy\'] for s in seeds for row in all_results[m][s]]\n        means_en[m] = (np.mean(en_vals), np.std(en_vals))\n    en_row = "| **Mean Total Energy (J)** | " + " | ".join([f"{means_en[m][0]:.2f} +/- {means_en[m][1]:.2f}" for m in models]) + " |"\n    print(en_row)\n    \n    means_rw = {}\n    for m in models:\n        rw_vals = [row[\'episode_reward\'] for s in seeds for row in all_results[m][s]]\n        means_rw[m] = (np.mean(rw_vals), np.std(rw_vals))\n    rw_row = "| **Mean Episode Reward** | " + " | ".join([f"{means_rw[m][0]:.2f} +/- {means_rw[m][1]:.2f}" for m in models]) + " |"\n    print(rw_row)\n    \n    if "quantum" in models and "classical" in models:\n        diff_abs = means_ee[\'quantum\'][0] - means_ee[\'classical\'][0]\n        diff_pct = (diff_abs / max(means_ee[\'classical\'][0], 1e-6)) * 100.0\n        print("-" * 80)\n        print(f"Absolute Difference (Quantum - Classical): {diff_abs:+.4f} Mb/J")\n        print(f"Relative Difference                     : {diff_pct:+.2f}%")\n        print("=" * 80 + "\\n")\n\ndef main():\n    parser = argparse.ArgumentParser(description="Controlled Quantum vs Classical QFRL Benchmark")\n    parser.add_argument("--config", type=str, default="configs/paper_config.yaml", help="Path to config YAML")\n    parser.add_argument("--episodes", type=int, default=15, help="Number of episodes per seed (default: 15)")\n    parser.add_argument("--steps", type=int, default=50, help="Steps per episode (default: 50)")\n    parser.add_argument("--seeds", nargs="+", type=int, default=[42, 123, 2026], help="Seeds to evaluate")\n    parser.add_argument("--models", nargs="+", type=str, default=["classical", "quantum"], help="Models: classical, quantum")\n    parser.add_argument("--device", type=str, default="cuda" if torch.cuda.is_available() else "cpu", help="Device (cpu/cuda)")\n    parser.add_argument("--output_dir", type=str, default="results", help="Directory for output metrics and plots")\n    parser.add_argument("--allow_mock", action="store_true", help="Allow mock TorchQuantum if running offline")\n    parser.add_argument("--pilot_only", action="store_true", help="Run only 2-episode pilot verification")\n    parser.add_argument("--noise_schedule", type=str, default="constant", choices=["constant", "decay"], help="Exploration schedule")\n    \n    args = parser.parse_args()\n    \n    with open(args.config, \'r\') as f:\n        cfg = yaml.safe_load(f)\n        \n    if args.pilot_only:\n        run_pilot_smoke_test(cfg, models=args.models, device=args.device, allow_mock=args.allow_mock)\n        return\n        \n    # First run pilot\n    pilot_ok = run_pilot_smoke_test(cfg, models=args.models, device=args.device, allow_mock=args.allow_mock)\n    if not pilot_ok:\n        print("[FAIL] Pilot smoke test failed. Aborting full benchmark.")\n        sys.exit(1)\n        \n    # Then run full benchmark\n    run_controlled_comparison(\n        config_path=args.config,\n        models=args.models,\n        episodes=args.episodes,\n        steps_per_episode=args.steps,\n        seeds=args.seeds,\n        device=args.device,\n        output_dir=args.output_dir,\n        allow_mock=args.allow_mock,\n        noise_schedule=args.noise_schedule\n    )\n\nif __name__ == "__main__":\n    main()\n'
ddpg_content = 'import torch\nimport torch.nn as nn\nimport torch.optim as optim\nimport numpy as np\n\nfrom quantum.qnn import QuantumActor\n\nclass Critic(nn.Module):\n    """\n    Classical Critic for QFRL.\n    \n    IMPLEMENTATION CHOICE:\n    The exact critic architecture is not paper-specified. \n    We use a configurable modest MLP.\n    Input: state (150) + action (5) = 155\n    Output: Q-value (1)\n    """\n    def __init__(self, state_dim=150, action_dim=5, hidden_dim1=64, hidden_dim2=64):\n        super(Critic, self).__init__()\n        \n        self.net = nn.Sequential(\n            nn.Linear(state_dim + action_dim, hidden_dim1),\n            nn.ReLU(),\n            nn.Linear(hidden_dim1, hidden_dim2),\n            nn.ReLU(),\n            nn.Linear(hidden_dim2, 1)\n        )\n        \n    def forward(self, state, action):\n        x = torch.cat([state, action], dim=-1)\n        return self.net(x)\n\nclass DDPGAgent:\n    """\n    Single-Client DDPG Agent for QFRL.\n    """\n    def __init__(self, state_dim=150, action_dim=5, config=None):\n        self.config = config or {}\n        self.device = torch.device(self.config.get(\'device\', \'cpu\'))\n        \n        # PAPER-SPECIFIED hyperparameters\n        self.gamma = self.config.get(\'gamma\', 0.95)\n        self.actor_lr = self.config.get(\'actor_lr\', 0.001)\n        self.critic_lr = self.config.get(\'critic_lr\', 0.0025)\n        \n        # IMPLEMENTATION CHOICE hyperparameters\n        self.tau = self.config.get(\'tau\', 0.005)\n        self.exploration_noise = self.config.get(\'exploration_noise\', 0.1)\n        \n        # Initialize Actor and Critic\n        self.actor = QuantumActor(state_dim=state_dim, action_dim=action_dim).to(self.device)\n        self.critic = Critic(state_dim=state_dim, action_dim=action_dim, \n                             hidden_dim1=self.config.get(\'critic_hidden1\', 64),\n                             hidden_dim2=self.config.get(\'critic_hidden2\', 64)).to(self.device)\n                             \n        self.actor_target = QuantumActor(state_dim=state_dim, action_dim=action_dim).to(self.device)\n        self.critic_target = Critic(state_dim=state_dim, action_dim=action_dim, \n                                    hidden_dim1=self.config.get(\'critic_hidden1\', 64),\n                                    hidden_dim2=self.config.get(\'critic_hidden2\', 64)).to(self.device)\n                                    \n        self.actor_target.load_state_dict(self.actor.state_dict())\n        self.critic_target.load_state_dict(self.critic.state_dict())\n        \n        # Optimizers\n        self.actor_optimizer = optim.Adam(self.actor.parameters(), lr=self.actor_lr)\n        self.critic_optimizer = optim.Adam(self.critic.parameters(), lr=self.critic_lr)\n        self.critic_criterion = nn.MSELoss()\n\n    def select_action(self, state: np.ndarray, add_noise=True, noise_scale=None) -> np.ndarray:\n        """Selects a continuous action in [-1, 1]."""\n        state_tensor = torch.FloatTensor(state).unsqueeze(0).to(self.device)\n        \n        with torch.no_grad():\n            action = self.actor(state_tensor).cpu().numpy()[0]\n            \n        if add_noise:\n            # IMPLEMENTATION CHOICE: Gaussian exploration noise\n            scale = noise_scale if noise_scale is not None else self.exploration_noise\n            noise = np.random.normal(0, scale, size=action.shape)\n            action = action + noise\n            \n        return np.clip(action, -1.0, 1.0)\n        \n    def train(self, replay_buffer, batch_size=32):\n        """Performs one step of DDPG training."""\n        if len(replay_buffer) < batch_size:\n            return None, None\n            \n        batch = replay_buffer.sample(batch_size)\n        \n        state = torch.FloatTensor(batch[\'states\']).to(self.device)\n        action = torch.FloatTensor(batch[\'actions\']).to(self.device)\n        reward = torch.FloatTensor(batch[\'rewards\']).to(self.device)\n        next_state = torch.FloatTensor(batch[\'next_states\']).to(self.device)\n        done = torch.FloatTensor(batch[\'dones\']).to(self.device)\n        \n        # Critic Update\n        with torch.no_grad():\n            next_action = self.actor_target(next_state)\n            target_q = self.critic_target(next_state, next_action)\n            target_q = reward + (1 - done) * self.gamma * target_q\n            \n        current_q = self.critic(state, action)\n        critic_loss = self.critic_criterion(current_q, target_q)\n        \n        self.critic_optimizer.zero_grad()\n        critic_loss.backward()\n        self.critic_optimizer.step()\n        \n        # Actor Update\n        actor_loss = -self.critic(state, self.actor(state)).mean()\n        \n        self.actor_optimizer.zero_grad()\n        actor_loss.backward()\n        self.actor_optimizer.step()\n        \n        # Soft Target Updates (IMPLEMENTATION CHOICE)\n        for param, target_param in zip(self.actor.parameters(), self.actor_target.parameters()):\n            target_param.data.copy_(self.tau * param.data + (1 - self.tau) * target_param.data)\n            \n        for param, target_param in zip(self.critic.parameters(), self.critic_target.parameters()):\n            target_param.data.copy_(self.tau * param.data + (1 - self.tau) * target_param.data)\n            \n        return actor_loss.item(), critic_loss.item()\n        \n    def save_checkpoint(self, path: str, episode: int, seed: int):\n        """Saves a checkpoint, excluding ephemeral simulation state buffers."""\n        actor_sd = {k: v for k, v in self.actor.state_dict().items() if not k.endswith(\'states\')}\n        target_actor_sd = {k: v for k, v in self.actor_target.state_dict().items() if not k.endswith(\'states\')}\n        torch.save({\n            \'episode\': episode,\n            \'seed\': seed,\n            \'actor_state_dict\': actor_sd,\n            \'critic_state_dict\': self.critic.state_dict(),\n            \'actor_target_state_dict\': target_actor_sd,\n            \'critic_target_state_dict\': self.critic_target.state_dict(),\n            \'actor_optimizer_state_dict\': self.actor_optimizer.state_dict(),\n            \'critic_optimizer_state_dict\': self.critic_optimizer.state_dict(),\n            \'config\': self.config\n        }, path)\n        \n    def load_checkpoint(self, path: str):\n        """Loads a checkpoint, safely ignoring ephemeral quantum state buffers."""\n        checkpoint = torch.load(path, map_location=self.device)\n        actor_sd = {k: v for k, v in checkpoint[\'actor_state_dict\'].items() if not k.endswith(\'states\')}\n        target_actor_sd = {k: v for k, v in checkpoint[\'actor_target_state_dict\'].items() if not k.endswith(\'states\')}\n        self.actor.load_state_dict(actor_sd, strict=False)\n        self.critic.load_state_dict(checkpoint[\'critic_state_dict\'])\n        self.actor_target.load_state_dict(target_actor_sd, strict=False)\n        self.critic_target.load_state_dict(checkpoint[\'critic_target_state_dict\'])\n        self.actor_optimizer.load_state_dict(checkpoint[\'actor_optimizer_state_dict\'])\n        self.critic_optimizer.load_state_dict(checkpoint[\'critic_optimizer_state_dict\'])\n        return checkpoint[\'episode\'], checkpoint[\'seed\'], checkpoint[\'config\']\n'

with open('rl/classical_actor.py', 'w', encoding='utf-8') as f:
    f.write(classical_actor_content)

with open('experiments/compare_quantum_classical.py', 'w', encoding='utf-8') as f:
    f.write(compare_content)

with open('rl/ddpg.py', 'w', encoding='utf-8') as f:
    f.write(ddpg_content)

print('Stage 9 benchmark code synced into repository.')


In [ ]:
# 4. Launch Controlled Matched Benchmark (Classical & Quantum across Seeds 42, 123, 2026)
# Runs pilot test first, then 15 episodes per seed per model (45 eps total each)
!python -u experiments/compare_quantum_classical.py \
    --models classical quantum \
    --episodes 15 \
    --steps 50 \
    --seeds 42 123 2026 \
    --device cuda \
    --output_dir results


In [ ]:
# 5. Export Results, Generate 1-Click ZIP Download, and Display Plot
import os, shutil
import pandas as pd
from IPython.display import Image, display, FileLink

src_results = '/kaggle/working/QRFL/QFRL_Highway/results'
dst_results = '/kaggle/working/results'
zip_output = '/kaggle/working/qfrl_stage9_results'

if os.path.exists(src_results):
    if os.path.exists(dst_results):
        shutil.rmtree(dst_results)
    shutil.copytree(src_results, dst_results)
    shutil.make_archive(zip_output, 'zip', src_results)
    
    print('SUCCESS! Download your Stage 9 Benchmark Results below:')
    display(FileLink('qfrl_stage9_results.zip'))
    
    plot_file = os.path.join(src_results, 'quantum_classical_comparison.png')
    if os.path.exists(plot_file):
        print('\nComparison Learning Curves:')
        display(Image(filename=plot_file, width=900))
        
    csv_file = os.path.join(src_results, 'quantum_classical_comparison.csv')
    if os.path.exists(csv_file):
        print('\nSummary of Comparison Metrics (Top 10 rows):')
        df = pd.read_csv(csv_file)
        display(df[['model_type', 'seed', 'episode', 'energy_efficiency_mbits_per_joule', 'total_energy', 'episode_reward']].head(10))
else:
    print('Results folder not found.')
